# physics-informed mri reconstruction colab

undersampled k-space -> high-fidelity image via an unrolled net with a data-consistency layer runs the whole pipeline on a colab accelerator smoke test -> train on synthetic -> fastmri -> evaluate + figures

> colab storage is ephemeral mount google drive and save the data subset checkpoints and figures there or they vanish when the runtime disconnects

set runtime to gpu via runtime -> change runtime type (t4/l4/a100/h100) see the table for which accelerator to pick

| accelerator | good for | notes |
|---|---|---|
| a100 / h100 | both projects fast | ampere/hopper tf32 + bf16 ideal for the unrolled net |
| l4 | both projects | ada 24 gb bf16/tf32 great value |
| t4 (free) | synthetic + small fastmri | 16 gb use amp_dtype fp16 (no bf16) |
| g4 / older gpu | synthetic + small runs | auto-detected via nvidia-smi |
| cpu | smoke test + synthetic | validates the pipeline too slow for real training |
| v5e-1 / v6e-1 tpu | project 2 only (maybe) | see the tpu note below |

> tpu note project 1 reconstruction is built on complex-valued ffts not reliably
> supported on tpu/xla use a gpu runtime for project 1 project 2 (real-valued
> segmentation) can in principle use a tpu but monai + torch_xla is finicky a gpu
> is still the smoother path

## 1. check the runtime / accelerator

In [ ]:
import os, subprocess, sys
print("Python:", sys.version.split()[0])
try:
    out = subprocess.check_output(
        ["nvidia-smi", "--query-gpu=name,memory.total,driver_version", "--format=csv,noheader"]
    ).decode().strip()
    print("GPU:", out)
except Exception:
    print("No NVIDIA GPU detected (CPU or TPU runtime).")
print("TPU env:", os.environ.get("COLAB_TPU_ADDR", "(none)"))

## 2. get the project code

pick one method below by setting SETUP
- github easiest once the repo is pushed (set REPO_URL)
- drive put the 01-mri-reconstruction folder in google drive (set PROJECT_DIR)
- upload a zip of the project

In [ ]:
SETUP = "github"   # "github" | "drive" | "upload"
REPO_URL = "https://github.com/cosmicquilt/mri-reconstruction.git"
PROJECT_DIR = "/content/mri-reconstruction"

import os
if SETUP == "github":
    if os.path.exists(PROJECT_DIR):
        get_ipython().system(f'git -C {PROJECT_DIR} pull')   # already cloned, pull latest
    else:
        get_ipython().system(f'git clone {REPO_URL} {PROJECT_DIR}')
elif SETUP == "drive":
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_DIR = "/content/drive/MyDrive/job/projects/01-mri-reconstruction"  # set me
elif SETUP == "upload":
    from google.colab import files
    import zipfile, io
    up = files.upload()  # choose a project .zip
    name = next(iter(up))
    with zipfile.ZipFile(io.BytesIO(up[name])) as z:
        z.extractall("/content")
    PROJECT_DIR = "/content/" + name.replace(".zip", "")

os.chdir(PROJECT_DIR)
import sys
SRC = os.path.join(PROJECT_DIR, "src")
sys.path.insert(0, SRC)                          # in-kernel imports (import mri_recon ...)
os.environ["PYTHONPATH"] = SRC + os.pathsep + os.environ.get("PYTHONPATH", "")  # for !python cells
print("project dir:", os.getcwd())

## 3. install + smoke test

colab already ships a cuda build of pytorch so install only the extra deps (torch left untouched) the code is already importable via the path set in the previous cell no editable install needed (that needs a kernel restart on colab)

In [ ]:
!pip -q install pyyaml h5py scikit-image matplotlib
!python scripts/smoke_test.py

## 4. device + gpu speedups

In [ ]:
from mri_recon.utils import get_device, configure_backend
device = get_device("auto")          # cuda if present else cpu
configure_backend(device)            # tf32 + cudnn autotune on ampere+
print("using device:", device)

## 5. train on synthetic data (runs on any runtime)

no download needed trains the unet and the unrolled net on synthetic phantoms in a few minutes proves the full pytorch path on the accelerator before real data

In [ ]:
# unet
!python -m mri_recon.cli train --config configs/synthetic.yaml --set train.epochs=8
# unrolled net (fp32 complex fft)
!python -m mri_recon.cli train --config configs/synthetic.yaml \
  --set model.name=unrolled train.batch_size=4 \
        output.dir=results/synthetic_unrolled train.epochs=8
# big ampere+ gpu? add  --set train.amp=true  to the unet line

## 6. evaluate + figures (synthetic)

In [ ]:
# all three methods, both checkpoints (list quoted so the shell doesnt glob it)
!python -m mri_recon.cli eval --config configs/eval.yaml \
  --set 'eval.methods=[zero_filled,unet,unrolled]' \
        eval.checkpoints.unet=results/synthetic/best.pt \
        eval.checkpoints.unrolled=results/synthetic_unrolled/best.pt
!python -m mri_recon.cli figures --config configs/eval.yaml \
  --set 'eval.methods=[zero_filled,unet,unrolled]' \
        eval.checkpoints.unet=results/synthetic/best.pt \
        eval.checkpoints.unrolled=results/synthetic_unrolled/best.pt

from IPython.display import Image, display
import glob
for p in sorted(glob.glob("results/eval/**/*.png", recursive=True)):
    print(p)
    display(Image(filename=p))

## 7. fastmri (knee single-coil val) - one-cell setup

download only knee_singlecoil_val (~15 gb) once and save the tar to drive (see scripts/download_fastmri.md the signed nyu urls are time-limited tokens dont commit them). this ONE cell mounts drive, (re)clones the code, extracts the tar to fast local disk, and defines DATA / OUT / PROJECT_DIR. safe to re-run after a disconnect, it skips work thats already done

In [ ]:
import os, sys, glob, subprocess
REPO_URL = "https://github.com/cosmicquilt/mri-reconstruction.git"
PROJECT_DIR = "/content/mri-reconstruction"
if not os.path.isdir(PROJECT_DIR):
    subprocess.run(['git', 'clone', '--quiet', REPO_URL, PROJECT_DIR], check=True)
subprocess.run(['git', '-C', PROJECT_DIR, 'pull', '--quiet'])
SRC = f"{PROJECT_DIR}/src"
if SRC not in sys.path: sys.path.insert(0, SRC)
os.environ["PYTHONPATH"] = SRC
from google.colab import drive
drive.mount("/content/drive")
DRIVE = '/content/drive/MyDrive/fastmri'        # the tar + your results live here
OUT = f"{DRIVE}/results"; os.makedirs(OUT, exist_ok=True)
tars = glob.glob(f"{DRIVE}/*.tar.xz") + glob.glob(f"{DRIVE}/*.tar")
print("tar on Drive:", tars)
if not glob.glob("/content/fastmri/**/*.h5", recursive=True):
    os.makedirs("/content/fastmri", exist_ok=True)
    print("extracting", tars[0], "..."); subprocess.run(["tar", "-xf", tars[0], "-C", "/content/fastmri"], check=True)
h5 = glob.glob("/content/fastmri/**/*.h5", recursive=True)
assert h5, "no .h5 under /content/fastmri - is the tar on Drive at $DRIVE?"
DATA = os.path.dirname(h5[0])
print("DATA =", DATA, "| num .h5 =", len(h5), "| OUT =", OUT)

## 8. the controlled comparison (matched loss, in-distribution)

trains the u-net and the unrolled net with the SAME l1 loss at the SAME acceleration, so the only variable is the architecture. honest finding (see the readme): at 8x single-coil a well-trained u-net beats the much smaller unrolled net. quick pass (epochs 40, 1200 slices), drop the overrides for a fuller run

In [ ]:
%cd $PROJECT_DIR
!python -m mri_recon.cli train --config configs/fastmri_knee_split.yaml \
  --set data.split_dir=$DATA 'mask.accelerations=[8]' 'mask.center_fractions=[0.04]' \
        output.dir=$OUT/unet_8x train.epochs=40 data.train_limit=1200
!python -m mri_recon.cli train --config configs/fastmri_knee_split.yaml \
  --set data.split_dir=$DATA model.name=unrolled train.batch_size=1 \
        'mask.accelerations=[8]' 'mask.center_fractions=[0.04]' \
        output.dir=$OUT/unrolled_8x train.epochs=40 data.train_limit=1200

## 9. evaluate (3-way, in-distribution 8x)

In [ ]:
!python -m mri_recon.cli eval --config configs/fastmri_knee_split.yaml \
  --set data.split_dir=$DATA output.dir=$OUT/eval 'eval.methods=[zero_filled,unet,unrolled]' \
        'eval.accelerations=[8]' 'eval.center_fractions=[0.04]' \
        eval.checkpoints.unet=$OUT/unet_8x/best.pt eval.checkpoints.unrolled=$OUT/unrolled_8x/best.pt
!python -m mri_recon.cli figures --config configs/fastmri_knee_split.yaml \
  --set data.split_dir=$DATA output.dir=$OUT/eval 'eval.methods=[zero_filled,unet,unrolled]' \
        'eval.accelerations=[8]' 'eval.center_fractions=[0.04]' \
        eval.checkpoints.unet=$OUT/unet_8x/best.pt eval.checkpoints.unrolled=$OUT/unrolled_8x/best.pt
from IPython.display import Image, display
import glob
for p in sorted(glob.glob(f"{OUT}/eval/**/*.png", recursive=True)):
    print(p); display(Image(filename=p))

## 10. downstream radiomic feature stability (bridge to biomarkers)

measures how each reconstruction perturbs the radiomic features a biomarker pipeline would compute. roi segmented ONCE on the ground truth and reused on every recon (isolates reconstruction from segmentation), 25 ibsi-style features (first-order + glcm, whole-image z-score, fixed bin width), agreement scored per feature by lin's ccc + icc(2,1) with a bootstrap ci and a paired wilcoxon. pyradiomics if it installs else a self-contained numpy extractor. single-seed here, see section 11 for the 3-seed + mixed-effects headline

In [ ]:
!pip -q install pyradiomics 2>/dev/null && echo "pyradiomics OK" || echo "pyradiomics unavailable -> numpy fallback"
!python -m mri_recon.cli radiomics --config configs/fastmri_knee_split.yaml \
  --set data.split_dir=$DATA 'eval.methods=[zero_filled,unet,unrolled]' \
        eval.checkpoints.unet=$OUT/unet_8x/best.pt eval.checkpoints.unrolled=$OUT/unrolled_8x/best.pt \
        radiomics.acceleration=8 radiomics.center_fraction=0.04 radiomics.limit=100 \
        output.dir=$OUT/radio8x

## 11. the rigorous version: 3 seeds + mixed-effects

the headline radiomics result trains the comparison across 3 seeds and aggregates the per-feature ccc with a conservative seeds-averaged paired wilcoxon plus a linear mixed-effects model (method fixed, seed fixed, random intercept per feature). heavier (3x training) but this is the publication-grade run. result: the unrolled preserves features significantly better (+0.037 ccc, p~1e-4, 19/25 features), texture-driven, despite losing on ssim/psnr

In [ ]:
%cd $PROJECT_DIR
for s in [1, 2, 3]:
    !python -m mri_recon.cli train --config configs/fastmri_knee_split.yaml --set data.split_dir=$DATA seed={s} 'mask.accelerations=[8]' 'mask.center_fractions=[0.04]' output.dir=$OUT/seed{s}/unet_8x train.epochs=40 data.train_limit=1200
    !python -m mri_recon.cli train --config configs/fastmri_knee_split.yaml --set data.split_dir=$DATA seed={s} model.name=unrolled train.batch_size=1 'mask.accelerations=[8]' 'mask.center_fractions=[0.04]' output.dir=$OUT/seed{s}/unrolled_8x train.epochs=40 data.train_limit=1200
    !python -m mri_recon.cli radiomics --config configs/fastmri_knee_split.yaml --set data.split_dir=$DATA 'eval.methods=[zero_filled,unet,unrolled]' eval.checkpoints.unet=$OUT/seed{s}/unet_8x/best.pt eval.checkpoints.unrolled=$OUT/seed{s}/unrolled_8x/best.pt radiomics.acceleration=8 radiomics.center_fraction=0.04 radiomics.limit=100 output.dir=$OUT/seed{s}/radio

In [ ]:
import pandas as pd, numpy as np
from scipy.stats import wilcoxon
import statsmodels.formula.api as smf
df = pd.concat([pd.read_csv(f'{OUT}/seed{s}/radio/radiomics/radiomics_per_feature.csv').assign(seed=s) for s in (1,2,3)], ignore_index=True)
df = df[df.method.isin(['unet','unrolled'])]
avg = df.groupby(['feature','method']).ccc.mean().unstack('method')
d = avg.unrolled - avg.unet
print('seeds-averaged, N features =', len(avg), '| mean diff =', round(float(d.mean()),4), '| wilcoxon p =', wilcoxon(avg.unrolled, avg.unet).pvalue, '|', int((d>=0).sum()), 'of', len(d), 'favor unrolled')
g = df.copy(); g['is_unrolled'] = (g.method=='unrolled').astype(int)
m = smf.mixedlm('ccc ~ is_unrolled + C(seed)', g, groups=g.feature).fit()
print('LME+seed: effect =', round(float(m.params['is_unrolled']),4), '| p =', float(m.pvalues['is_unrolled']))

## accelerator cheatsheet
- batch size scales with vram bump --set train.batch_size=... on a100/h100 keep 1 for the unrolled net on real (variable-size) fastmri k-space
- bf16 (a100/h100/l4) is the safe fast path fp16 for t4
- mixed precision is unet-only here the unrolled model complex fft stays fp32
- tpu use a gpu for this project (complex fft isnt xla-friendly)